# Build Khmer TTS token shards — Kaggle

Turns the DDD Khmer corpus (~128GB of audio) into **ready-to-train VQ token
shards** (~1.7GB total) and publishes them to your Hugging Face **dataset** repo.

## Why
Training reads *only* Fish Speech's protobuf shards — never the audio. Measured
on this corpus those shards are **~451 bytes per second of audio, ~71x smaller
than the WAVs**. So you pay the expensive preprocessing + GPU token-extraction
cost **once**, and every future training session pulls ~1.7GB and trains
immediately — no download, no denoise/VAD/resample, no VQ pass.

## How it stays inside Kaggle's limits
- `/kaggle/working` is 20GB and the corpus is ~128GB, so parquet files are
  fetched **one at a time**, decoded, and **deleted** — the parquet footprint
  never exceeds a single file (~113MB).
- Every `VQ_BATCH_CLIPS` clips are flushed into a shard, uploaded, and wiped
  from local disk.
- **Progress lives in the HF repo**, not on local disk. A 12h timeout costs you
  only the shard in flight: just re-run and it continues where it stopped.

## What to do
1. Settings ▸ Accelerator ▸ **GPU T4 x2**  (both GPUs get used)
2. Settings ▸ **Internet ▸ On**
3. Add-ons ▸ Secrets ▸ add **`HF_TOKEN`** with **write** access
4. Run All. Section 6 runs a **pilot** and prints measured throughput plus an
   ETA — read it before starting the real run in Section 7.
5. Re-run the notebook each session until it prints `NOTHING LEFT TO DO`.


## 0 · Config

In [ ]:
# ============================== CONFIG ==============================
GITHUB_URL   = "https://github.com/Pich09/voice-clone.git"
SRC_DATASET  = "DDD-Cambodia/khmer-speech-dataset"   # source audio corpus
SHARD_REPO   = "Panhapich/DDD-speech-shard"          # target: YOUR dataset repo

# Source parquet processed before a shard is cut + uploaded. This is *source
# data read*, not disk used -- files are downloaded and deleted one by one.
CHUNK_GB        = 15

# Clips buffered before each extract_vq pass. Bounds peak disk: 20000 clips is
# roughly 48h of audio = ~5.6GB of 16kHz WAV, comfortable inside the 20GB quota.
VQ_BATCH_CLIPS  = 20000

# Kaggle sessions die at 12h. Stop cleanly before that so the in-flight shard
# still uploads (checked at shard boundaries).
MAX_HOURS       = 10.5

# extract_vq spawns this many workers and round-robins them across visible
# GPUs, so 4 keeps both T4s busy. Drop the batch size if you hit CUDA OOM.
EXTRACT_WORKERS    = 4
EXTRACT_BATCH_SIZE = 16

PILOT_CLIPS     = 2000   # Section 6: measure throughput, upload nothing
# ====================================================================
print("Config loaded.")

## 1 · Environment

In [ ]:
import os, subprocess, sys, shutil

IN_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print("Kaggle:", IN_KAGGLE)

WORKDIR = "/kaggle/working/khmer-voice-clone" if IN_KAGGLE else os.getcwd()

# Parquet scratch: prefer /kaggle/temp, which is NOT part of the 20GB
# /kaggle/working quota. Falls back to working/ if it is unavailable.
PARQUET_DIR = "/kaggle/temp/ddd_parquet" if IN_KAGGLE else "/tmp/ddd_parquet"
try:
    os.makedirs(PARQUET_DIR, exist_ok=True)
    with open(os.path.join(PARQUET_DIR, ".w"), "w") as f: f.write("ok")
    os.remove(os.path.join(PARQUET_DIR, ".w"))
except Exception as e:
    PARQUET_DIR = os.path.join(WORKDIR, "ddd_parquet")
    print("!! /kaggle/temp unusable, falling back to", PARQUET_DIR, f"({e})")
print("parquet scratch:", PARQUET_DIR)

def report_disk(label=""):
    t, u, f = shutil.disk_usage("/kaggle/working" if IN_KAGGLE else ".")
    print(f"[disk{' - ' + label if label else ''}] used={u/1e9:.1f}GB "
          f"free={f/1e9:.1f}GB of {t/1e9:.1f}GB")

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.device_count(), "GPU(s)")
for i in range(torch.cuda.device_count()):
    print("   ", i, torch.cuda.get_device_name(i))
if torch.cuda.device_count() < 2:
    print("!! Only one GPU visible -- set Accelerator to GPU T4 x2 to halve "
          "the VQ-extraction time.")
report_disk("start")

## 2 · Code (your repo + Fish Speech)

In [ ]:
if os.path.isdir(os.path.join(WORKDIR, "khmer_tts")):
    print("reusing", WORKDIR)
    if os.path.isdir(os.path.join(WORKDIR, ".git")):
        r = subprocess.run(["git", "-C", WORKDIR, "pull"], capture_output=True, text=True)
        print(r.stdout.strip() or r.stderr.strip())
else:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
print("cwd:", os.getcwd())
print("commit:", subprocess.run(["git","log","--oneline","-1"],
                                capture_output=True, text=True).stdout.strip())

FISH_DIR = os.path.join(WORKDIR, "vendor", "fish-speech")
if not os.path.isdir(os.path.join(FISH_DIR, "fish_speech")):
    shutil.rmtree(FISH_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/fishaudio/fish-speech", FISH_DIR], check=True)
print("fish-speech:", FISH_DIR)

## 3 · Dependencies

Only what `extract_vq` + `build_dataset` need — no training stack. Installed
one package at a time: pip aborts an entire multi-package command when any
single package fails, and a per-package failure at least names the culprit.

In [ ]:
def pip_install(args, required=True):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                       capture_output=True, text=True)
    if r.returncode != 0:
        msg = (f"pip install failed for {args}\n--- stdout ---\n{r.stdout[-2000:]}"
               f"\n--- stderr ---\n{r.stderr[-2000:]}")
        if required:
            raise RuntimeError(msg)
        print("!! optional install failed:", args); print(msg)
    return r.returncode == 0

pip_install(["huggingface_hub"])
# hf_xet's CDN path 403s intermittently ("invalid key pair id"); removing it
# forces plain HTTP downloads, which are reliable here.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "hf_xet"],
               capture_output=True, text=True)

import re, torch
_m = re.match(r"([0-9.]+)(?:\+(\w+))?", torch.__version__)
_ta = ["--no-deps", "--force-reinstall", f"torchaudio=={_m.group(1)}"]
if _m.group(2):
    _ta += ["--index-url", f"https://download.pytorch.org/whl/{_m.group(2)}"]
pip_install(_ta)
print("torchaudio pinned to torch", torch.__version__)

pip_install(["--no-deps", "-e", "vendor/fish-speech"])
for p in ["hydra-core", "loguru", "natsort", "einops", "rich", "click",
          "pyrootutils", "einx[torch]", "zstandard", "tiktoken", "cachetools",
          "safetensors", "pyloudnorm", "khmer-nltk", "soundfile", "pyarrow",
          "protobuf==4.25.5", "transformers==4.56.1"]:
    pip_install([p])

# DAC codec chain MUST be --no-deps: descript-audiotools pins protobuf<3.20,
# which conflicts with the protobuf pin above and sends pip backtracking
# through ancient sdists until one dies at `setup.py egg_info`. Both ship pure
# wheels; their real runtime deps are installed explicitly after.
pip_install(["--no-deps", "descript-audio-codec==1.0.0", "descript-audiotools==0.7.2"])
for p in ["argbind", "julius", "ffmpy", "flatten-dict", "markdown2",
          "randomname", "pystoi", "torch-stoi", "importlib-resources",
          "matplotlib"]:
    pip_install([p], required=False)
print("dependencies installed.")

## 4 · HF token (needs **write** access)

In [ ]:
HF_TOKEN = ""
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle Secrets.")
    except Exception as e:
        print("!! no HF_TOKEN secret:", e)
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from huggingface_hub import get_token
        HF_TOKEN = get_token() or ""
    except Exception:
        pass
if not HF_TOKEN:
    raise SystemExit("A write-scoped HF_TOKEN is required to publish shards. "
                     "Add-ons > Secrets > add HF_TOKEN.")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

from huggingface_hub import HfApi
me = HfApi(token=HF_TOKEN).whoami()
print("authenticated as:", me.get("name"), "| token role:",
      (me.get("auth") or {}).get("accessToken", {}).get("role", "?"))

## 5 · Codec checkpoint

Only `codec.pth` is needed — `extract_vq` uses the codec alone. Skipping
`model.pth` saves a 1.7GB download and 1.7GB of disk.

In [ ]:
from huggingface_hub import hf_hub_download
CKPT_DIR = os.path.join(WORKDIR, "checkpoints", "openaudio-s1-mini")
os.makedirs(CKPT_DIR, exist_ok=True)
codec = os.path.join(CKPT_DIR, "codec.pth")
if not os.path.isfile(codec):
    p = hf_hub_download("fishaudio/openaudio-s1-mini", "codec.pth",
                        token=HF_TOKEN, local_dir=CKPT_DIR)
    print("downloaded", p)
print("codec:", codec, f"{os.path.getsize(codec)/1e9:.2f}GB")
report_disk("after codec")

## 6 · Pilot — measure before committing GPU hours

Processes ~`PILOT_CLIPS` clips end to end and **uploads nothing**. Read the
`throughput` and ETA lines at the bottom. If throughput looks bad, lower
`EXTRACT_BATCH_SIZE` and re-run this cell before starting Section 7.

In [ ]:
cmd = [sys.executable, "scripts/15_build_token_shards.py",
       "--dataset", SRC_DATASET, "--repo", SHARD_REPO,
       "--pilot", str(PILOT_CLIPS),
       "--parquet-dir", PARQUET_DIR,
       "--extract-workers", str(EXTRACT_WORKERS),
       "--extract-batch-size", str(EXTRACT_BATCH_SIZE)]
print(" ".join(cmd), "\n")
rc = subprocess.run(cmd).returncode
report_disk("after pilot")
if rc != 0:
    raise RuntimeError(f"pilot failed (exit {rc}) -- see output above")

## 7 · Build shards for real

Publishes to `SHARD_REPO` as it goes and stops cleanly at `MAX_HOURS`.
**Re-run this cell (or the whole notebook) each session** — it resumes from
`progress.json` in the repo and skips everything already done. Done when it
prints `NOTHING LEFT TO DO`.

In [ ]:
cmd = [sys.executable, "scripts/15_build_token_shards.py",
       "--dataset", SRC_DATASET, "--repo", SHARD_REPO,
       "--chunk-gb", str(CHUNK_GB),
       "--vq-batch-clips", str(VQ_BATCH_CLIPS),
       "--max-hours", str(MAX_HOURS),
       "--parquet-dir", PARQUET_DIR,
       "--extract-workers", str(EXTRACT_WORKERS),
       "--extract-batch-size", str(EXTRACT_BATCH_SIZE)]
print(" ".join(cmd), "\n")
rc = subprocess.run(cmd).returncode
report_disk("after build")
if rc != 0:
    raise RuntimeError(f"build failed (exit {rc}) -- everything already "
                       "published is safe; re-run to continue")

## 8 · Progress

In [ ]:
from huggingface_hub import hf_hub_download
import json
try:
    fp = hf_hub_download(SHARD_REPO, "progress.json", repo_type="dataset",
                         token=HF_TOKEN, force_download=True)
    prog = json.load(open(fp))
    total_h = 1014.0     # measured corpus size (see repo notes)
    pct = 100 * prog["hours"] / total_h
    print(f"shards published : {len(prog['shards'])}")
    print(f"clips kept       : {prog['clips']:,}")
    print(f"audio processed  : {prog['hours']:.1f}h of ~{total_h:.0f}h ({pct:.1f}%)")
    print(f"source files done: {len(prog['processed_files'])}")
    print(f"rejected by QC   : {prog.get('rejected', 0):,}")
    print(f"duplicates       : {prog.get('duplicates', 0):,}")
    b = sum(s.get("proto_bytes", 0) for s in prog["shards"])
    print(f"shard bytes      : {b/1e6:.0f}MB")
    print(f"\nhttps://huggingface.co/datasets/{SHARD_REPO}")
    if pct < 99:
        print("\nNot finished -- re-run Section 7 in a new session to continue.")
    else:
        print("\nDone. Point training at this repo instead of the audio pipeline.")
except Exception as e:
    print("no progress.json yet (nothing published):", e)